# 03 — Corrélations avec les annotations médecin

**Objectif :** mesurer dans quelle mesure les métriques automatiques (MEDCON, COMET-QE, mBERT, Sentence-BERT) corrèlent avec le jugement humain d'un médecin sur la qualité de traduction.


In [ ]:
import subprocess, os
subprocess.run(['pip', 'install', '-q', '--upgrade', 'numpy'])
os.kill(os.getpid(), 9)

## 0. Setup

In [1]:
import os

if not os.path.exists('/content/Evaluating_medical_machine_translation'):
    !git clone https://github.com/11Maroua/Evaluating_medical_machine_translation.git

os.chdir('/content/Evaluating_medical_machine_translation')
print(f'Répertoire : {os.getcwd()}')

Répertoire : /content/Evaluating_medical_machine_translation


In [2]:
from google.colab import files

os.makedirs('data', exist_ok=True)

# Uploader : dr_annotations.json + cleaned_mesh_snomed_dico.json
uploaded = files.upload()
for fname in uploaded:
    dest = f'data/{fname}'
    os.replace(fname, dest)
    print(f'  → {dest}')

Saving cleaned_mesh_snomed_dico.json to cleaned_mesh_snomed_dico.json
Saving dr_annotations.json to dr_annotations.json
  → data/cleaned_mesh_snomed_dico.json
  → data/dr_annotations.json


In [2]:
!pip install -q pyahocorasick unbabel-comet sentence-transformers

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tensorflow 2.20.0 requires protobuf>=5.28.0, but you have protobuf 4.25.9 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 

## 1. Imports

In [4]:
import sys
import os
import json
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from scipy.stats import pearsonr, spearmanr

os.chdir('/content/Evaluating_medical_machine_translation')
sys.path.insert(0, 'src')
from medcon import build_pairs, build_automaton, medcon_grouped

print('Imports OK')

Imports OK


## 2. Chargement des annotations médecin

In [5]:
with open('data/dr_annotations.json', encoding='utf-8') as f:
    annotations_raw = json.load(f)

medical_data = []
for doc in annotations_raw:
    if not doc['annotations']:
        continue
    ann = doc['annotations'][0]
    score, errors = None, []
    for r in ann['result']:
        if r['from_name'] == 'translation_likert':
            score = int(r['value']['choices'][0])
        elif r['from_name'] == 'document_issue_types':
            errors = r['value']['choices']
    medical_data.append({
        'source_en'      : doc['data']['question'],
        'translation_fr' : doc['data']['translation'],
        'medical_score'  : score,
        'has_term_error' : 'Terminologie incohérente' in errors,
    })

print(f'Documents annotés   : {len(medical_data)}')
print(f'Score médecin moyen : {np.mean([d["medical_score"] for d in medical_data if d["medical_score"]]):.2f}/5')
print(f'Avec erreur termo   : {sum(d["has_term_error"] for d in medical_data)} docs')

Documents annotés   : 25
Score médecin moyen : 4.32/5
Avec erreur termo   : 21 docs


In [6]:
from collections import Counter

# Distribution des scores
scores = [d['medical_score'] for d in medical_data if d['medical_score']]
score_counts = Counter(scores)

print('=== DISTRIBUTION DES SCORES MÉDECIN ===\n')
for s in sorted(score_counts):
    bar = '█' * score_counts[s]
    print(f'  {s}/5 : {bar} ({score_counts[s]} docs)')
print(f'\n  Moyenne : {np.mean(scores):.2f}  |  Min : {min(scores)}  |  Max : {max(scores)}')

# Distribution des types d'erreurs
print('\n=== TYPES D\'ERREURS ===\n')
all_errors = []
for doc in annotations_raw:
    if not doc['annotations']:
        continue
    for r in doc['annotations'][0]['result']:
        if r['from_name'] == 'document_issue_types':
            all_errors.extend(r['value']['choices'])

error_counts = Counter(all_errors)
for error, count in error_counts.most_common():
    print(f'  {error:<35} : {count} docs ({100*count/len(medical_data):.0f}%)')

print(f'\n  Docs sans erreur : {len(medical_data) - len([d for d in medical_data if any([d["has_term_error"]])])}')

=== DISTRIBUTION DES SCORES MÉDECIN ===

  4/5 : █████████████████ (17 docs)
  5/5 : ████████ (8 docs)

  Moyenne : 4.32  |  Min : 4  |  Max : 5

=== TYPES D'ERREURS ===

  Terminologie incohérente            : 21 docs (84%)
  Grammaire/Style                     : 19 docs (76%)
  Contresens                          : 1 docs (4%)

  Docs sans erreur : 4


## 3. MEDCON sur les annotations médecin

In [7]:
with open('data/cleaned_mesh_snomed_dico.json', encoding='utf-8') as f:
    raw_dict = json.load(f)

pairs        = build_pairs(raw_dict)
automaton_en = build_automaton(pairs, 'en')
automaton_fr = build_automaton(pairs, 'fr')

print('Calcul MEDCON...')
for doc in tqdm(medical_data, desc='MEDCON'):
    r = medcon_grouped(doc['source_en'], doc['translation_fr'], pairs, automaton_en, automaton_fr)
    doc['medcon_f1']        = r['f1']
    doc['medcon_precision'] = r['precision']
    doc['medcon_recall']    = r['recall']

df_medical = pd.DataFrame(medical_data)
print(f'MEDCON F1 moyen : {df_medical["medcon_f1"].mean():.3f}')

Calcul MEDCON...


MEDCON:   0%|          | 0/25 [00:00<?, ?it/s]

MEDCON F1 moyen : 0.428


## 4. COMET-QE (reference-free)

In [8]:
USE_COMET_QE = False
try:
    import torch
    from comet import download_model, load_from_checkpoint

    for model_name in ['Unbabel/wmt20-comet-qe-da', 'Unbabel/eamt22-cometinho-da']:
        try:
            print(f'Tentative : {model_name}')
            comet_qe_model = load_from_checkpoint(download_model(model_name))
            print('OK')
            break
        except:
            continue

    comet_data = [{'src': d['source_en'], 'mt': d['translation_fr']} for d in medical_data]
    comet_output = comet_qe_model.predict(
        comet_data,
        batch_size=4,
        gpus=1 if torch.cuda.is_available() else 0
    )
    for i, doc in enumerate(medical_data):
        doc['comet_qe'] = float(comet_output.scores[i])

    df_medical   = pd.DataFrame(medical_data)
    USE_COMET_QE = True
    print(f'COMET-QE moyen : {df_medical["comet_qe"].mean():.3f}')

except Exception as e:
    print(f'COMET-QE indisponible : {e}')
    for doc in medical_data:
        doc['comet_qe'] = None
    df_medical = pd.DataFrame(medical_data)

COMET-QE indisponible : cannot import name 'runtime_version' from 'google.protobuf' (/usr/local/lib/python3.12/dist-packages/google/protobuf/__init__.py)


## 5. mBERT similarity

In [9]:
USE_MBERT = False
try:
    import torch
    from transformers import BertTokenizer, BertModel
    from sklearn.metrics.pairwise import cosine_similarity

    print('Chargement mBERT...')
    mbert_tok   = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
    mbert_model = BertModel.from_pretrained('bert-base-multilingual-cased')
    mbert_model.eval()

    def get_mbert_emb(text):
        inputs = mbert_tok(text, return_tensors='pt', max_length=512, truncation=True, padding=True)
        with torch.no_grad():
            out = mbert_model(**inputs)
        return out.last_hidden_state.mean(dim=1).squeeze().numpy()

    for doc in tqdm(medical_data, desc='mBERT'):
        sim = cosine_similarity(
            [get_mbert_emb(doc['source_en'])],
            [get_mbert_emb(doc['translation_fr'])]
        )[0][0]
        doc['mbert_sim'] = float(sim)

    df_medical = pd.DataFrame(medical_data)
    USE_MBERT  = True
    print(f'mBERT similarity moyen : {df_medical["mbert_sim"].mean():.3f}')

except Exception as e:
    print(f'mBERT indisponible : {e}')
    for doc in medical_data:
        doc['mbert_sim'] = None
    df_medical = pd.DataFrame(medical_data)

mBERT indisponible : cannot import name 'runtime_version' from 'google.protobuf' (/usr/local/lib/python3.12/dist-packages/google/protobuf/__init__.py)


## 6. Sentence-BERT similarity

In [10]:
USE_SBERT = False
try:
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity

    print('Chargement Sentence-BERT...')
    sbert = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

    for doc in tqdm(medical_data, desc='Sentence-BERT'):
        sim = cosine_similarity(
            [sbert.encode(doc['source_en'])],
            [sbert.encode(doc['translation_fr'])]
        )[0][0]
        doc['sbert_sim'] = float(sim)

    df_medical = pd.DataFrame(medical_data)
    USE_SBERT  = True
    print(f'Sentence-BERT similarity moyen : {df_medical["sbert_sim"].mean():.3f}')

except Exception as e:
    print(f'Sentence-BERT indisponible : {e}')
    for doc in medical_data:
        doc['sbert_sim'] = None
    df_medical = pd.DataFrame(medical_data)

Sentence-BERT indisponible : cannot import name 'runtime_version' from 'google.protobuf' (/usr/local/lib/python3.12/dist-packages/google/protobuf/__init__.py)


## 7. Corrélations avec le score médecin

In [11]:
print('=' * 80)
print('CORRÉLATIONS AVEC LE SCORE MÉDECIN')
print('=' * 80 + '\n')

valid_mask = df_medical['medical_score'].notna()
y = df_medical.loc[valid_mask, 'medical_score']

metrics_corr = [('MEDCON F1', 'medcon_f1')]
if USE_COMET_QE : metrics_corr.append(('COMET-QE',      'comet_qe'))
if USE_MBERT    : metrics_corr.append(('mBERT',         'mbert_sim'))
if USE_SBERT    : metrics_corr.append(('Sentence-BERT', 'sbert_sim'))

print(f"{'Métrique':<18} {'Pearson r':>10} {'p-val':>8} {'Spearman rho':>14} {'p-val':>8}")
print('-' * 64)

correlations = []
for name, col in metrics_corr:
    if col in df_medical.columns and df_medical.loc[valid_mask, col].notna().all():
        x = df_medical.loc[valid_mask, col]
        r_p, p_p = pearsonr(y, x)
        r_s, p_s = spearmanr(y, x)
        print(f'{name:<18} {r_p:>10.3f} {p_p:>8.4f} {r_s:>14.3f} {p_s:>8.4f}')
        correlations.append((name, col, r_p, p_p, r_s, p_s))


CORRÉLATIONS AVEC LE SCORE MÉDECIN

Métrique            Pearson r    p-val   Spearman rho    p-val
----------------------------------------------------------------
MEDCON F1               0.157   0.4524          0.134   0.5242


In [13]:
from scipy.stats import pointbiserialr

# Construire les vecteurs sur les docs qui ont un score médecin ET un MEDCON calculé
valid = [
    (d['has_term_error'], d['medcon_f1'])
    for d in medical_data
    if d['medical_score'] is not None and 'medcon_f1' in d
]

term_error_binary = [int(v[0]) for v in valid]  # 0 ou 1
medcon_f1_scores  = [v[1] for v in valid]

r_pb, p_pb = pointbiserialr(term_error_binary, medcon_f1_scores)

print('CORRÉLATION POINT-BISÉRIELLE')
print('Terminologie incohérente (0/1) vs MEDCON-like F1')
print('=' * 70)
print(f'\n  n = {len(valid)} documents')
print(f'  Docs avec erreur terminologie : {sum(term_error_binary)} / {len(valid)}')
print(f'\n  Point-biserial r : {r_pb:+.3f}')
print(f'  p-value          : {p_pb:.4f}')
print(f'\n  {"Corrélation significative (p < 0.05)" if p_pb < 0.05 else "Pas de corrélation significative (p >= 0.05)"}')

# Moyennes MEDCON-like par groupe
medcon_with_error    = [medcon_f1_scores[i] for i in range(len(valid)) if term_error_binary[i] == 1]
medcon_without_error = [medcon_f1_scores[i] for i in range(len(valid)) if term_error_binary[i] == 0]

print(f'\n  MEDCON-like F1 moyen (avec erreur termo)    : {np.mean(medcon_with_error):.3f}  (n={len(medcon_with_error)})')
print(f'  MEDCON-like F1 moyen (sans erreur termo)    : {np.mean(medcon_without_error):.3f}  (n={len(medcon_without_error)})')

CORRÉLATION POINT-BISÉRIELLE
Terminologie incohérente (0/1) vs MEDCON-like F1

  n = 25 documents
  Docs avec erreur terminologie : 21 / 25

  Point-biserial r : +0.179
  p-value          : 0.3912

  Pas de corrélation significative (p >= 0.05)

  MEDCON-like F1 moyen (avec erreur termo)    : 0.462  (n=21)
  MEDCON-like F1 moyen (sans erreur termo)    : 0.250  (n=4)
